In [1]:
import json 
import os 
import time 
from typing import Literal

import ollama
import pandas as pd 
from dotenv import load_dotenv
from openai import ContentFilterFinishReasonError, OpenAI, RateLimitError
from pydantic import BaseModel, Field 

In [2]:
load_dotenv()
openrouter_api_key=os.getenv("OPENROUTER_API_KEY")
openrouter_base_url="https://openrouter.ai/api/v1"
openrouter=OpenAI(base_url=openrouter_base_url, api_key=openrouter_api_key)


In [12]:
from loader import load_json, load_all
claims_path="E:\\undercover\\data\\evals\\held_out_claims.json"
labels_path="E:\\undercover\\data\\evals\\held_out_decisions.json"
policy_path="E:\\undercover\\data\\policies.json"
claims=load_json(claims_path)
labels=load_json(labels_path)
policies=load_json(policy_path)

len(claims), len(labels), len(policies)

(25, 25, 50)

In [15]:
policies={policy['policy_id']: policy for policy in policies}

In [16]:
labels={label['claim_id']: label for label in labels}


In [17]:
claims={claim['claim_id']: claim for claim in claims}



In [76]:
claims=claims.values()
labels=labels.values()
policies=policies.values()
#claims_list=list(claims)
#labels_list=list(labels)

In [81]:
claims

dict_values([{'claim_id': 'CLM-1101', 'policy_id': 'POL-2101', 'customer_id': 'CUS-3101', 'claim_type': 'trip_delay', 'submitted_on': '2026-08-24', 'incident_date': '2026-07-08', 'incident_location': 'Hong Kong', 'summary': 'Typhoon canceled a flight on a round the world trip, 26 hour delay', 'story': 'I am traveling around the world on a fixed schedule of eighty days with my traveling companion, Mr. Passepartout. On July 8 our flight from Hong Kong to Tokyo, from where we were to continue to Yokohama, was canceled because of a typhoon. We departed 26 hours later. I am claiming the hotel, our meals and the rebooking fee, which come to $580. I am also claiming $2,000 for a wager with members of my club in Boston, which I forfeited because the delay made me late for a stage of the journey we had agreed on. I keep precise records and every receipt is attached.', 'items': [{'description': 'Hotel in Hong Kong, one night, two travelers', 'category': 'accommodation', 'amount': 310.0}, {'descr

In [80]:
policies

dict_values([{'policy_id': 'POL-2001', 'customer_id': 'CUS-3001', 'plan': 'premium', 'add_ons': [], 'purchased_on': '2026-04-22', 'sold_through': 'Meridian Air checkout', 'trip': {'destination': 'Rome, Italy', 'departs_on': '2026-06-10', 'returns_on': '2026-06-18', 'trip_cost': 3100.0}, 'travelers': ['Maya Patel'], 'status': 'active'}, {'policy_id': 'POL-2002', 'customer_id': 'CUS-3002', 'plan': 'premium', 'add_ons': [], 'purchased_on': '2026-05-11', 'sold_through': 'Northwind Airlines checkout', 'trip': {'destination': 'Prague, Czech Republic', 'departs_on': '2026-06-20', 'returns_on': '2026-06-29', 'trip_cost': 2650.0}, 'travelers': ['Tom Becker'], 'status': 'active'}, {'policy_id': 'POL-2003', 'customer_id': 'CUS-3003', 'plan': 'premium', 'add_ons': [], 'purchased_on': '2026-05-04', 'sold_through': 'TripNest checkout', 'trip': {'destination': 'Lisbon, Portugal', 'departs_on': '2026-06-22', 'returns_on': '2026-07-01', 'trip_cost': 2900.0}, 'travelers': ['Aiko Tanaka'], 'status': 'act

In [82]:
claims_df=pd.DataFrame(claims)
labels_df=pd.DataFrame(labels)
policies_df=pd.DataFrame(policies)

In [83]:
policies_df.head(2)

,policy_id,customer_id,plan,add_ons,purchased_on,sold_through,trip,travelers,status
0,POL-2001,CUS-3001,premium,[],2026-04-22,Meridian Air checkout,"{'destination': 'Rome, Italy', 'departs_on': '...",[Maya Patel],active
1,POL-2002,CUS-3002,premium,[],2026-05-11,Northwind Airlines checkout,"{'destination': 'Prague, Czech Republic', 'dep...",[Tom Becker],active


In [58]:
claims_df.columns

Index(['claim_id', 'policy_id', 'customer_id', 'claim_type', 'submitted_on',
       'incident_date', 'incident_location', 'summary', 'story', 'items',
       'amount_claimed', 'documents', 'status'],
      dtype='str')

In [56]:
decisions=pd.merge(claims_df, labels_df, on='claim_id', how='inner')

In [59]:
decisions.decision.value_counts()

decision
approve    14
refer       6
decline     5
Name: count, dtype: int64

In [4]:
from pathlib import Path
import re
import pandas as pd

def split_policy_markdown(md_text: str) -> dict:
    """Split policy wording into { '1. Your plan': content, ... }"""
    # Find all ## headings
    pattern = re.compile(r'^##\s+(?P<num>\d+)\.\s+(?P<title>.+)\s*$', re.MULTILINE)
    matches = list(pattern.finditer(md_text))
    
    sections = {}
    # intro before first ##
    if matches:
        intro = md_text[:matches[0].start()].strip()
        if intro:
            sections["0. Intro"] = intro

    for i, m in enumerate(matches):
        key = f"{m.group('num')}. {m.group('title').strip()}"
        start = m.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(md_text)
        content = md_text[start:end].strip()
        sections[key] = content
    return sections


In [5]:
md = Path("data/policy_wording.md").read_text(encoding="utf-8")
sections = split_policy_markdown(md)
sections

{'0. Intro': '# UnderCover Travel Insurance: Policy Wording\n\nVersion 2026.1, for policies bought on or after January 1, 2026.\n\nThis document explains what your UnderCover policy covers, what it leaves out and how we work out what we pay. Your policy record shows your plan, any add-ons you bought, the dates of your covered trip and the travelers on it. All amounts are in US dollars.',
 '1. Your plan': '| Benefit | Economy | Premium | First |\n|---|---|---|---|\n| Trip Cancellation and Interruption, maximum | $1,500 | $5,000 | $10,000 |\n| Trip Delay, waiting period | 12 hours | 6 hours | 3 hours |\n| Trip Delay, maximum | $300 | $800 | $1,500 |\n| Baggage Delay, waiting period | 24 hours | 12 hours | 6 hours |\n| Baggage Delay, maximum | $100 | $300 | $500 |\n| Baggage and Personal Effects, maximum | $750 | $1,500 | $3,000 |\n| Per-item limit | $150 | $300 | $500 |\n| Passport and Travel Documents, maximum | $150 | $250 | $400 |\n| Emergency Medical, maximum | $25,000 | $100,000 | $

In [6]:
INSTRUCTIONS=""" You are a claims handler at undercover travel insurance.
Decide the claim you are given, using the policy wording below and the 
customers policy record.

Choose one decision:
approve: the wording covers the claim and the required documents are attached. Workout the payout using the payment section
decline: the wording does not cover the claim.
refer: a colleage needs to look at it because a required document is missing or the information disagree with the documents provided
"""
def make_user_prompt(claim):
    policy=policies[claim['policy_id']]
    record={key: value for key, value in policy.items() if key not in ["policy_id", "customer_id","sold_through","status"]}
    details={key:value for key, value in claim.items() if key not in ["policy_id","customer_id","status"]}
    details["documents"]=[{"name":document["name"], "type": document["type"]} for document in claim["documents"]]
    return f"""# Policy record
    
    {json.dumps(record)}

#claim

     {json.dumps(details)}


"""
def make_messages(claim):
    return [
        {"role":"system","content":INSTRUCTIONS},
        {'role': "user", "content":make_user_prompt(claim)}
    ]


In [27]:
list(claims.values())[0]

{'claim_id': 'CLM-1101',
 'policy_id': 'POL-2101',
 'customer_id': 'CUS-3101',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-08-24',
 'incident_date': '2026-07-08',
 'incident_location': 'Hong Kong',
 'summary': 'Typhoon canceled a flight on a round the world trip, 26 hour delay',
 'story': 'I am traveling around the world on a fixed schedule of eighty days with my traveling companion, Mr. Passepartout. On July 8 our flight from Hong Kong to Tokyo, from where we were to continue to Yokohama, was canceled because of a typhoon. We departed 26 hours later. I am claiming the hotel, our meals and the rebooking fee, which come to $580. I am also claiming $2,000 for a wager with members of my club in Boston, which I forfeited because the delay made me late for a stage of the journey we had agreed on. I keep precise records and every receipt is attached.',
 'items': [{'description': 'Hotel in Hong Kong, one night, two travelers',
   'category': 'accommodation',
   'amount': 310.0},
  {'d

In [47]:
make_messages(list(claims.values())[0])

[{'role': 'system',
  'content': ' You are a claims handler at undercover travel insurance.\nDecide the claim you are given, using the policy wording below and the \ncustomers policy record.\n\nChoose one decision:\napprove: the wording covers the claim and the required documents are attached. Workout the payout using the payment section\ndecline: the wording does not cover the claim.\nrefer: a colleage needs to look at it because a required document is missing or the information disagree with the documents provided\n'},
 {'role': 'user',
  'content': '# Policy record\n\n    {"plan": "first", "add_ons": [], "purchased_on": "2026-04-15", "trip": {"destination": "Round the world: London, Suez, Mumbai, Kolkata, Hong Kong, Yokohama, San Francisco, New York", "departs_on": "2026-06-01", "returns_on": "2026-08-19", "trip_cost": 18400.0}, "travelers": ["Phil Fogg", "Jean Passepartout"]}\n\n#claim\n\n     {"claim_id": "CLM-1101", "claim_type": "trip_delay", "submitted_on": "2026-08-24", "incid

In [30]:
class ClaimDecision(BaseModel):
    rationale: str=Field(description="2-3 sentences to explain the decision")
    decision: Literal['approve','decline','refer']
    payout: float=Field(description="Approved payout in USD; 0 for decline; null for refer")

def decide(claim, model, effort):
    provider={'require_parameters':True} #tells open router to a provider that supports everything in the request.
    response=openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={"provider":provider}
    )
    answer=response.choices[0].message.parsed
    return{
        "claim_id":claim['claim_id'],
        "decision":answer.decision,
        "payout":answer.payout,
        "rationale": answer.rationale
    }

In [32]:
first=decide(list(claims.values())[0],'deepseek/deepseek-v4.1-flash','low')

In [33]:
first

{'claim_id': 'CLM-1101',
 'decision': 'refer',
 'payout': 0.0,
 'rationale': 'The trip-delay items for hotel, meals, and rebooking are generally the type of expenses covered under trip delay, but the $2,000 forfeited club wager is a consequential loss and would not be covered. However, the claim includes a $150 rebooking fee while the attached documents list does not include a rebooking fee receipt or proof of payment. Because a required supporting document is missing, this needs referral to a colleague for review before a final decision can be made.'}

In [34]:
def is_right(answer, label):
    return answer['decision']==label['decision'] and abs(answer['payout']- label["payout"])<=1

In [227]:
!pip install tqdm

In [229]:
len(claims), len(labels)

(25, 25)

In [48]:
claims_l=list(claims.values())

In [50]:
for claim in claims_l:
    print(claim)

{'claim_id': 'CLM-1101', 'policy_id': 'POL-2101', 'customer_id': 'CUS-3101', 'claim_type': 'trip_delay', 'submitted_on': '2026-08-24', 'incident_date': '2026-07-08', 'incident_location': 'Hong Kong', 'summary': 'Typhoon canceled a flight on a round the world trip, 26 hour delay', 'story': 'I am traveling around the world on a fixed schedule of eighty days with my traveling companion, Mr. Passepartout. On July 8 our flight from Hong Kong to Tokyo, from where we were to continue to Yokohama, was canceled because of a typhoon. We departed 26 hours later. I am claiming the hotel, our meals and the rebooking fee, which come to $580. I am also claiming $2,000 for a wager with members of my club in Boston, which I forfeited because the delay made me late for a stage of the journey we had agreed on. I keep precise records and every receipt is attached.', 'items': [{'description': 'Hotel in Hong Kong, one night, two travelers', 'category': 'accommodation', 'amount': 310.0}, {'description': 'Mea

In [51]:
from tqdm import tqdm
def run_all(model, effort, claims, labels):
    results=[]
    for claim in tqdm(claims):
        answer=decide(claim, model, effort)
        results.append(is_right(answer, labels[claim['claim_id']]))
    return results

In [52]:
results=run_all('deepseek/deepseek-v4.1-flash','low',claims_l, labels)

100%|██████████| 25/25 [05:48<00:00, 13.92s/it]


In [54]:
#Ollama
import ollama
print(ollama.pull('gemma4:e2b'))

status='success' completed=None total=None digest=None


In [63]:
def decide_ollama(claim, model):
    response=ollama.chat(
        model=model,
        messages=make_messages(claim),
        format=ClaimDecision.model_json_schema(),
        think=False,
        options={"num_ctx":8192}
    )
    answer=ClaimDecision.model_validate_json(response.message.content)
    return{
            "claim_id":claim['claim_id'],
            "decision":answer.decision,
            "payout":answer.payout,
            "rationale": answer.rationale
        }

def run_all(model, claims, labels):
    results=[]
    for claim in tqdm(claims):
        answer=decide_ollama(claim, model)
        results.append(is_right(answer, labels[claim['claim_id']]))
    return results

In [61]:
resp=decide_ollama(claims_l[0],'gemma4:e2b')

In [67]:
claims_l[0]

{'claim_id': 'CLM-1101',
 'policy_id': 'POL-2101',
 'customer_id': 'CUS-3101',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-08-24',
 'incident_date': '2026-07-08',
 'incident_location': 'Hong Kong',
 'summary': 'Typhoon canceled a flight on a round the world trip, 26 hour delay',
 'story': 'I am traveling around the world on a fixed schedule of eighty days with my traveling companion, Mr. Passepartout. On July 8 our flight from Hong Kong to Tokyo, from where we were to continue to Yokohama, was canceled because of a typhoon. We departed 26 hours later. I am claiming the hotel, our meals and the rebooking fee, which come to $580. I am also claiming $2,000 for a wager with members of my club in Boston, which I forfeited because the delay made me late for a stage of the journey we had agreed on. I keep precise records and every receipt is attached.',
 'items': [{'description': 'Hotel in Hong Kong, one night, two travelers',
   'category': 'accommodation',
   'amount': 310.0},
  {'d

In [69]:
resp_all=run_all('gemma4:e2b', claims_l, labels)

100%|██████████| 25/25 [00:21<00:00,  1.18it/s]
